---
### __1.1 Loading__
---

In [ ]:
import pandas as pd

In [ ]:
caminho_arquivo = '../data/raw/MiningProcess_Flotation_Plant_Database.csv'

df = pd.read_csv(caminho_arquivo, sep=',', decimal=',', parse_dates=["date"])

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.head(10)

---
### __1.2 Temporal structure__
---

In [ ]:
# Records per hour
rows_per_hour = df.groupby('date').size()
print(rows_per_hour.value_counts().sort_index())
print(rows_per_hour[rows_per_hour != 180])

In [ ]:
# Missing hours
full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="h")

missing = full_range.difference(rows_per_hour.index)

print(f"expected: {len(full_range)}, present: {len(rows_per_hour)}, missing: {len(missing)}")

missing_s = missing.to_series()
block_id = missing_s.diff().ne(pd.Timedelta(hours=1)).cumsum()
blocks = missing_s.groupby(block_id.values).agg(["min", "max", "count"])
print(blocks)


In [ ]:
# Column that vary

max_distinct_per_hour = df.groupby("date").nunique().max().sort_values()
print(max_distinct_per_hour)

In [ ]:
target = "% Silica Concentrate"
distinct_target = df.groupby("date")[target].nunique()
varying = distinct_target[distinct_target > 1]

print(f"hours with varying target: {len(varying)}")
print(f"from {varying.index.min()} to {varying.index.max()}")

In [ ]:
cols = ['% Iron Concentrate', '% Iron Feed', '% Silica Feed', '% Silica Concentrate', 'Flotation Column 01 Level', 'Flotation Column 03 Air Flow']
mask = (df['date'] >= '2017-04-11 01:00:00') & (df['date'] < '2017-04-21 02:00:00')

s_1 = df.loc[mask]
#diffs = s_1[cols].diff()
diffs = s_1.diff()

print((diffs != 0).sum())           # quantas vezes cada coluna mudou na hora


In [ ]:
diffs.abs().describe()       # tamanho das variações

In [ ]:
step = df.groupby("date")[target].diff()
step_std = step.groupby(df["date"]).std()

print(step_std.loc[varying.index].describe())

In [ ]:
hourly = df.groupby("date")[target].agg(["first", "last"])
next_first = hourly["first"].shift(-1, freq="h").reindex(hourly.index)
gap_to_next = (hourly["last"] - next_first).abs()

print("varying hours:")
print(gap_to_next.loc[varying.index].describe())
print("constant hours (reference):")
print(gap_to_next.drop(varying.index).describe())

In [ ]:
import matplotlib.pyplot as plt

sample_hours = varying.index[[0, len(varying) // 2, -1]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, hour in zip(axes, sample_hours):
    window = df[df["date"].between(hour - pd.Timedelta(hours=1), hour + pd.Timedelta(hours=1))]
    ax.plot(window[target].to_numpy())
    ax.set_title(str(hour))
    ax.set_xlabel("row (20 s each)")
    ax.set_ylabel(target)
plt.tight_layout()

### Finding: interpolated target values (data leakage risk)

The lab assay for `% Silica Concentrate` is reported once per hour, so within an hour
all 180 rows (20 s each) should carry the same value. In **310 hours (~7.5%)**, spread
from 2017-03-12 to 2017-09-06, the target changes on every row instead.

**Evidence that these values are linear interpolations toward the next hour:**

| Check | Interpolated hours (n = 310) | Regular hours (n = 3,785) |
|---|---|---|
| Std. dev. of row-to-row steps within the hour | max $5.0 \times 10^{-11}$ (constant step) | — |
| \|last value of hour *h* − first value of hour *h*+1\| (median) | 0.0008 | 0.23 |

- **Shape:** the step between consecutive rows is constant within each hour, i.e.
  the values lie on a straight line.
- **Destination:** each ramp ends at the value of the next hour, roughly 300× closer
  than the typical hour-to-hour jump between real assays.
- **Blocks:** interpolated hours appear in consecutive runs (e.g. a single straight
  ramp spanning three hours around 2017-05-16 19:00), consistent with missing lab
  results being filled by connecting the last known assay to the next one.

**Implication:** in these hours the target is not a measurement. It was built using
the *next* lab result, which did not exist at prediction time. Training or evaluating
on these rows leaks future information, and chronological splits do not catch it
because the leak is inside each row. How to handle them (exclude or flag) is decided
in section 1.6, together with the target definition.

In [ ]:
feed = "% Iron Feed"

hourly_feed = df.groupby("date")[feed].first().asfreq("h")
block_id = hourly_feed.ne(hourly_feed.shift()).cumsum()


runs = hourly_feed.groupby(block_id.values).agg(
    start=lambda s: s.index.min(),
    hours="size",
    value="first",
)
runs = runs[runs["value"].notna()]

print(f"blocks: {len(runs)}")
print(runs["hours"].describe())
print(runs["hours"].value_counts().sort_index().head(15))


------------------------
__Finding__
- To be analysed in next phases as a feature: “hours since last feed updating"
------------------------

In [ ]:
print(runs.loc[runs["hours"].isin([6, 9]), "start"].dt.hour.value_counts().sort_index())
print(runs[runs["hours"] > 48])

---
__Finding__

- Standard time feed: 01:00/ 07:00/ 16:00
- Same feed value(64.03) per 33 days: between 2017-05-13 01:00:00 - 2017-06-14 00:00:00
---

---
### __1.3 Data quality__
---

In [ ]:
# Check Nulls

nulls = df.isna().sum()
print(nulls[nulls > 0] if nulls.any() else "no nulls")

In [ ]:
# Check duplicated

print(f"extra copies: {df.duplicated().sum()}")
print(f"rows involved in duplicates: {df.duplicated(keep=False).sum()}")

consecutive = df.eq(df.shift()).all(axis=1)
print(f"rows identical to the previous row: {consecutive.sum()}")

print(df.loc[df.duplicated(), "date"].dt.to_period("M").value_counts().sort_index())

In [ ]:
# Validate business rules x data

range_rules = {
    # assays (%)
    "% Iron Feed": (0, 100),
    "% Silica Feed": (0, 100),
    "% Iron Concentrate": (0, 70),  # hematite (Fe2O3) is ~69.9% Fe
    "% Silica Concentrate": (0, 100),
    # reagents and pulp
    "Starch Flow": (0, None),  # flows cannot be negative
    "Amina Flow": (0, None),
    "Ore Pulp Flow": (0, None),
    "Ore Pulp pH": (8, 11),  # reverse cationic flotation runs ~10
    "Ore Pulp Density": (1.0, 3.0),  # pulp denser than water, below solids density
}
for i in range(1, 8):
    range_rules[f"Flotation Column 0{i} Air Flow"] = (0, None)
    range_rules[f"Flotation Column 0{i} Level"] = (0, None)


def count_range_violations(frame: pd.DataFrame, rules: dict) -> pd.DataFrame:
    result = {}
    for col, (low, high) in rules.items():
        values = frame[col]
        bad = pd.Series(False, index=values.index)
        if low is not None:
            bad |= values < low
        if high is not None:
            bad |= values > high
        result[col] = {
            "low": low,
            "high": high,
            "violations": int(bad.sum()),
            "pct": round(100 * bad.mean(), 3),
        }
    return pd.DataFrame(result).T


print(count_range_violations(df, range_rules))

cross_rules = {
    "feed: Fe + SiO2 > 100": df["% Iron Feed"] + df["% Silica Feed"] > 100,
    "concentrate: Fe + SiO2 > 100": df["% Iron Concentrate"] + df["% Silica Concentrate"] > 100,
}
print(pd.Series({name: int(mask.sum()) for name, mask in cross_rules.items()}, name="violations"))

In [ ]:
# Check sensor lock

fast_cols = [c for c in df.columns if c not in ("date", "% Iron Feed", "% Silica Feed")]


def longest_run(values: pd.Series) -> int:
    block_id = values.ne(values.shift()).cumsum()
    return int(values.groupby(block_id).size().max())


stuck = pd.DataFrame({"longest_run_rows": {c: longest_run(df[c]) for c in fast_cols}})
stuck["longest_run_hours"] = (stuck["longest_run_rows"] / 180).round(1)
print(stuck.sort_values("longest_run_rows", ascending=False))

In [ ]:
# Search event in the time

def stuck_runs(frame: pd.DataFrame, col: str, min_rows: int) -> pd.DataFrame:
    values = frame[col]
    block_id = values.ne(values.shift()).cumsum()
    runs = frame.groupby(block_id.values).agg(
        start=("date", "min"),
        end=("date", "max"),
        rows=("date", "size"),
    )
    runs = runs[runs["rows"] >= min_rows].reset_index(drop=True)
    runs.insert(0, "column", col)
    return runs


fast_process_cols = [c for c in fast_cols if c not in ("% Iron Concentrate", "% Silica Concentrate")]

events = pd.concat(
    [stuck_runs(df, c, min_rows=360) for c in fast_process_cols]
    + [stuck_runs(df, c, min_rows=180 * 24) for c in ("% Iron Concentrate", "% Silica Concentrate")],
    ignore_index=True,
)
events["hours"] = (events["rows"] / 180).round(1)
print(events.sort_values(["start", "column"]).to_string())

---
### Finding: plausible but false values

No nulls and no physically impossible values were found: all range rules (assays
between 0 and 100%, non-negative flows, pH 8–11, pulp density 1–3) and cross-column
rules (Fe + SiO₂ ≤ 100%) pass. The quality problems in this dataset are values that
look valid but were not measured.

**Frozen acquisition (whole plant).** On 2017-07-11 06:00–10:00 (3.7 h) all 19 process
variables hold the same value simultaneously. This accounts for most of the 1,171
duplicated rows, all of which are consecutive and fall almost entirely in June–July.

**Frozen sensor groups.** Subsets of sensors freeze together, suggesting a shared
network or I/O card:

| Period | Sensors | Duration |
|---|---|---|
| 2017-03-10 → 2017-04-11 11:00 | Air flow, columns 04 and 05 | ~460 h (from dataset start) |
| 2017-06-21 14:00–17:00 | Level 01, air flow 06 and 07, pH | 2.5 h |
| 2017-07-31 11:00–16:00 | pH | 5.1 h |
| 2017-08-02 08:00–15:00 | Air flow 01, 02, 03, 06, 07, pH | 7.7 h |

**Repeated lab assays.** Both concentrate assays (Fe and SiO₂) stay constant for
28–74 consecutive hours in three periods (2017-03-31, 2017-07-31, 2017-08-17),
suggesting missing results were forward-filled. Unlike the interpolated hours
(section 1.2), this does not leak future information, but it creates labels that were
never measured.

**Implication:** each pattern becomes a data quality rule in the silver layer
(stage 2): flag frozen runs per sensor, flag whole-row freezes, and flag stale lab
assays.
---